In [1]:
import torch
from torchvision.models import mobilenet_v3_large as mobilenet
from torchvision.models import MobileNet_V3_Large_Weights as pre_weights

from sklearn.metrics import confusion_matrix, accuracy_score
import math

import helpers.data_prefetcher as data_prefetcher
import helpers.data_handler as data_handler
import helpers.utils as utils

### System details

In [2]:
print(f"PyTorch version: {torch.__version__}")

print("--------------------------------------------------")
print(f"Using cuda: {torch.cuda.is_available()}")
print(f"Cuda corrent device: {torch.cuda.current_device()}")
print(f"Cuda device: {torch.cuda.get_device_name(torch.cuda.current_device())}")
print(f"Torch Backend enable: {torch.backends.cudnn.enabled}")
print(f"Torch Backend: {torch.backends.cudnn.version() }")

PyTorch version: 2.2.1+cu121
--------------------------------------------------
Using cuda: True
Cuda corrent device: 0
Cuda device: NVIDIA GeForce GTX 1660 SUPER
Torch Backend enable: True
Torch Backend: 8902


In [3]:
NUM_CLASSES = 1
NUM_EPOCHS = 10

### Getting data

In [4]:
train, validation, test = data_handler.get_datasets()
print(f"Train dataset size: {len(train)}")
print(f"Validation dataset size: {len(validation)}")
print(f"Test dataset size: {len(test)}")

Train dataset size: 174817
Validation dataset size: 96811
Test dataset size: 424223


In [5]:
collate_fn = lambda batch: utils.fast_collate(batch)

train_loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True, num_workers=6, collate_fn=collate_fn, pin_memory=True)
val_loader = torch.utils.data.DataLoader(validation, batch_size=1000, shuffle=False, num_workers=6, collate_fn=collate_fn, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=1000, shuffle=False, num_workers=6, collate_fn=collate_fn, pin_memory=True)

### Fine tunning

In [6]:
model = mobilenet(weights=pre_weights.IMAGENET1K_V2)
model.classifier[-1] = torch.nn.Linear(1280, NUM_CLASSES)

bce_loss = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model.classifier)
print(f"is in cuda: {next(model.parameters()).is_cuda}")
print(f"device: {device}")

Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=1, bias=True)
)
is in cuda: True
device: cuda


In [7]:
best_state_dict = None
best_loss = math.inf

In [8]:
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    prefetcher = data_prefetcher.data_prefetcher(train_loader)
    inputs, labels = prefetcher.next()
    while inputs is not None:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        preds = model(inputs).squeeze(1)
        loss = bce_loss(preds, labels.float())

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        inputs, labels = prefetcher.next()
    
    epoch_loss = running_loss / len(train)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}], Training Loss: {epoch_loss:.10f}")

    # Validation 
    model.eval()
    val_labels, val_preds, val_loss = [], [], 0.0
    val_bce_loss = torch.nn.BCEWithLogitsLoss()

    prefetcher = data_prefetcher.data_prefetcher(val_loader)
    inputs, labels = prefetcher.next()
    with torch.no_grad():
        while inputs is not None:
            inputs, labels = inputs.to(device), labels.to(device)
            preds = model(inputs)

            loss = val_bce_loss(preds.squeeze(1), labels.float())
            val_loss += loss.item() * inputs.size(0)

            val_labels.extend(labels.cpu().numpy())
            val_preds.extend((torch.sigmoid(preds).cpu().numpy() > 0.5).astype(int))

            inputs, labels = prefetcher.next()

    val_loss = val_loss / len(validation)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}], Validation Loss: {val_loss:.10f}")

    if (best_loss - val_loss) > 0.0:
        best_state_dict = model.state_dict()
        best_loss = val_loss

    accuracy = accuracy_score(val_labels, val_preds)
    cm = confusion_matrix(val_labels, val_preds)
    
    print(f'Validation Accuracy: {accuracy:.2f}%\n')
    utils.print_confusion_matrix(cm)
    print("----------------------------------------------------------------------")

Epoch [1/10], Training Loss: 0.0113657573
Epoch [1/10], Validation Loss: 0.0147286249
Validation Accuracy: 1.00%

Confusion matrix:
66299 113 
248 30151 
----------------------------------------------------------------------
Epoch [2/10], Training Loss: 0.0057246109
Epoch [2/10], Validation Loss: 0.0177282336
Validation Accuracy: 0.99%

Confusion matrix:
65952 460 
34 30365 
----------------------------------------------------------------------
Epoch [3/10], Training Loss: 0.0100035040
Epoch [3/10], Validation Loss: 0.0127263493
Validation Accuracy: 1.00%

Confusion matrix:
66231 181 
80 30319 
----------------------------------------------------------------------
Epoch [4/10], Training Loss: 0.0039932945
Epoch [4/10], Validation Loss: 0.0246647810
Validation Accuracy: 0.99%

Confusion matrix:
65640 772 
10 30389 
----------------------------------------------------------------------
Epoch [5/10], Training Loss: 0.0036961167
Epoch [5/10], Validation Loss: 0.0547554505
Validation Accura

### Results

In [9]:
final_model = mobilenet()
final_model.classifier[-1] = torch.nn.Linear(1280, NUM_CLASSES)
final_model.load_state_dict(best_state_dict)
final_model.to(device)

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [10]:
final_model.eval()

test_labels = []
test_preds = []

prefetcher = data_prefetcher.data_prefetcher(test_loader)
inputs, labels = prefetcher.next()
with torch.no_grad():
    while inputs is not None:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = final_model(inputs)

        test_labels.extend(labels.cpu().numpy())
        test_preds.extend((torch.sigmoid(preds).cpu().numpy() > 0.5).astype(int))

        inputs, labels = prefetcher.next()

accuracy = accuracy_score(test_labels, test_preds)
cm = confusion_matrix(test_labels, test_preds)

print(f'Test Accuracy: {accuracy:.2f}%\n')
utils.print_confusion_matrix(cm)

Test Accuracy: 0.99%

Confusion matrix:
189480 4244 
930 229569 


In [15]:
final_model.state_dict()

OrderedDict([('features.0.0.weight',
              tensor([[[[ 1.0007e-01, -3.4770e-02,  1.2713e-01],
                        [-2.1172e-01, -7.4409e-02,  3.1166e-01],
                        [-1.9541e-01, -3.5087e-01, -2.4921e-02]],
              
                       [[ 7.2119e-01,  7.3657e-01,  7.9686e-01],
                        [ 1.3545e-01,  1.3440e-01,  5.1463e-01],
                        [ 1.9186e-01,  2.5791e-03,  4.0569e-01]],
              
                       [[ 2.4573e-01,  2.0809e-01,  1.8470e-01],
                        [-1.3644e-01,  1.0724e-01,  2.2839e-01],
                        [-1.7661e-01, -2.4269e-01, -9.6757e-02]]],
              
              
                      [[[-1.9516e-01,  7.5482e-02,  5.0334e-02],
                        [-1.3126e-02,  1.2490e+00, -1.0652e+00],
                        [ 6.5717e-02,  1.7207e+00, -1.7773e+00]],
              
                       [[-3.7588e-01,  2.4706e-01, -3.7823e-02],
                        [ 2.4875e-02, 

In [16]:
model.state_dict()

OrderedDict([('features.0.0.weight',
              tensor([[[[ 1.0007e-01, -3.4770e-02,  1.2713e-01],
                        [-2.1172e-01, -7.4409e-02,  3.1166e-01],
                        [-1.9541e-01, -3.5087e-01, -2.4921e-02]],
              
                       [[ 7.2119e-01,  7.3657e-01,  7.9686e-01],
                        [ 1.3545e-01,  1.3440e-01,  5.1463e-01],
                        [ 1.9186e-01,  2.5791e-03,  4.0569e-01]],
              
                       [[ 2.4573e-01,  2.0809e-01,  1.8470e-01],
                        [-1.3644e-01,  1.0724e-01,  2.2839e-01],
                        [-1.7661e-01, -2.4269e-01, -9.6757e-02]]],
              
              
                      [[[-1.9516e-01,  7.5482e-02,  5.0334e-02],
                        [-1.3126e-02,  1.2490e+00, -1.0652e+00],
                        [ 6.5717e-02,  1.7207e+00, -1.7773e+00]],
              
                       [[-3.7588e-01,  2.4706e-01, -3.7823e-02],
                        [ 2.4875e-02, 